### Install Required Packages
---

In [1]:
!pip install requests

### Hugging Face API Configuration
---

In [2]:
from openai import OpenAI

api_key  = "YOUR_OPENROUTER_API_KEY"
base_url = "https://openrouter.ai/api/v1"

client = OpenAI(base_url=base_url, api_key=api_key)

### Translation Function
---

In [3]:
def translate_arabic_to_english(arabic_text):
    response = client.chat.completions.create(
        model="openai/gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "You are a professional Arabic to English translator. Translate accurately and naturally."},
            {"role": "user",   "content": f"Translate the following Arabic text to English:\n{arabic_text}"}
        ],
        max_tokens=300,
        temperature=0.3
    )
    return response.choices[0].message.content.strip()

### Test the Translation Function
---

In [4]:
## Test with sample Arabic sentences:
test_sentences = [
    "ما هي أسباب الحرب العالمية الأولى؟",
    "يعاني المريض من ارتفاع ضغط الدم",
    "أنا أدرس الذكاء الاصطناعي في الجامعة",
    "كيف يمكنني تحسين مهاراتي في البرمجة؟",
    "ما هو الاقتصاد الدائري؟"
]

for sentence in test_sentences:
    translation = translate_arabic_to_english(sentence)
    print(f"Arabic  : {sentence}")
    print(f"English : {translation}")
    print()

Arabic  : ما هي أسباب الحرب العالمية الأولى؟
English : What are the causes of World War I?

Arabic  : يعاني المريض من ارتفاع ضغط الدم
English : The patient suffers from high blood pressure.

Arabic  : أنا أدرس الذكاء الاصطناعي في الجامعة
English : I study artificial intelligence at the university.

Arabic  : كيف يمكنني تحسين مهاراتي في البرمجة؟
English : How can I improve my programming skills?

Arabic  : ما هو الاقتصاد الدائري؟
English : What is the circular economy?



### Translate Questions and Answers from AAFAQ Dataset
---

In [7]:
import pandas as pd

## Load the preprocessed dataset:
df = pd.read_csv('/content/AAFAQ_Dataset.csv', encoding='utf-8-sig')
df.head()

,QuestionText,Category,Answer
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,التعليم,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...
1,أليس القطن عماد الثروة في مصر؟,الاقتصاد والعمل,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...
2,أتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تُعرف بأنها كائنات حية دقيقة.
4,أيتكون الهواء أساساً من النيتروجين؟,التعليم,الهواء يتكون أساساً من النيتروجين.


In [8]:
## Translate a sample of questions and answers from the dataset:
sample = df.sample(5, random_state=42)

for _, row in sample.iterrows():
    print(f"Question (AR) : {row['QuestionText']}")
    print(f"Question (EN) : {translate_arabic_to_english(row['QuestionText'])}")
    print(f"Answer   (AR) : {row['Answer'][:100]}")
    print(f"Answer   (EN) : {translate_arabic_to_english(row['Answer'][:100])}")
    print(f"Category      : {row['Category']}")
    print("-" * 60)

Question (AR) : علامَ تعتمدُ في تحقيقِ أهدافِك؟
Question (EN) : What do you rely on to achieve your goals?
Answer   (AR) : أعتمد في تحقيق أهدافي على التخطيط الجيد والعمل الجاد والاستمرار في التعلم والتطوير الشخصي.
Answer   (EN) : I rely on good planning, hard work, continuous learning, and personal development to achieve my goals.
Category      : الثقافة
------------------------------------------------------------
Question (AR) : من أهم الدول المستوردة للمنتجات الزراعية الأردنية؟
Question (EN) : What are the most important countries importing Jordanian agricultural products?
Answer   (AR) : يعتبر القطاع الزراعي من القطاعات المهمة في الأردن، وتلعب الزراعة دورا هاما في المنظومة الاقتصادية وا
Answer   (EN) : The agricultural sector is considered one of the important sectors in Jordan, and agriculture plays a significant role in the economic system.
Category      : السياسة والقانون
------------------------------------------------------------
Question (AR) : ما هي أهمية التعليم العالي؟
Ques

### Translate with Category Context (Prompt Modification)
---

In [12]:
## Advanced prompt: include category context for better translation quality:
def translate_with_context(arabic_text, category):
    response = client.chat.completions.create(
        model="openai/gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": f"You are a professional Arabic to English translator specializing in {category} topics."},
            {"role": "user",   "content": f"Translate the following Arabic text to English:\n{arabic_text}"}
        ],
        max_tokens=300,
        temperature=0.3
    )
    return response.choices[0].message.content.strip()

    response = requests.post(API_URL, headers=headers, json=payload)
    result   = response.json()

    if "choices" in result:
        return result["choices"][0]["message"]["content"].strip()
    else:
        return f"Error: {result}"

In [13]:
## Test with category context:
for _, row in sample.iterrows():
    translation = translate_with_context(row['QuestionText'], row['Category'])
    print(f"Category : {row['Category']}")
    print(f"Arabic   : {row['QuestionText']}")
    print(f"English  : {translation}")
    print()

Category : الثقافة
Arabic   : علامَ تعتمدُ في تحقيقِ أهدافِك؟
English  : What do you rely on to achieve your goals?

Category : السياسة والقانون
Arabic   : من أهم الدول المستوردة للمنتجات الزراعية الأردنية؟
English  : What are the most important countries importing Jordanian agricultural products?

Category : التعليم
Arabic   : ما هي أهمية التعليم العالي؟
English  : What is the importance of higher education?

Category : العلوم
Arabic   : لماذا الأكسجين مهم؟
English  : Why is oxygen important?

Category : التعليم
Arabic   : أي مادة تُعتبر موصل جيد للكهرباء؟
English  : Which material is considered a good conductor of electricity?

